# 1.2.3 Distribution Sampling Exercise

Implements TODO 1.2.3: common distributions + sampling exercise.

Goals:
1. Sample from each distribution in `notes/L1-foundations/10-common-distributions.md`.
2. Visualize empirical samples.
3. Compare empirical mean/variance to theoretical values.
4. Attach each distribution to a biomaterials/biomedical use case.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bernoulli, binom, norm, gamma, beta, multivariate_normal

rng = np.random.default_rng(42)
N = 1000

plt.style.use('seaborn-v0_8-whitegrid')
print(f'Sample size per distribution: {N}')

## Helper plotting utilities

In [ ]:
def report_moments(name, samples, theo_mean, theo_var):
    emp_mean = float(np.mean(samples))
    emp_var = float(np.var(samples, ddof=0))
    print(f"{name}:")
    print(f"  empirical mean = {emp_mean:.4f}, theoretical mean = {theo_mean:.4f}")
    print(f"  empirical var  = {emp_var:.4f}, theoretical var  = {theo_var:.4f}")
    print()

def plot_discrete(samples, xs, pmf, title):
    counts = np.bincount(samples.astype(int), minlength=int(max(xs))+1)
    probs = counts / counts.sum()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(xs, probs[xs], alpha=0.6, label='empirical PMF')
    ax.plot(xs, pmf, 'o-', color='black', label='theoretical PMF')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('probability')
    ax.legend()
    plt.show()

def plot_continuous(samples, x_grid, pdf, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(samples, bins=40, density=True, alpha=0.6, label='empirical density')
    ax.plot(x_grid, pdf, color='black', linewidth=2, label='theoretical PDF')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('density')
    ax.legend()
    plt.show()

## Bernoulli(p)
Use case: binary event such as scaffold sample failure (yes/no).

In [ ]:
p = 0.30
samples_ber = rng.binomial(1, p, size=N)
xs = np.array([0, 1])
pmf = bernoulli.pmf(xs, p)
plot_discrete(samples_ber, xs, pmf, f'Bernoulli(p={p})')
report_moments('Bernoulli', samples_ber, theo_mean=p, theo_var=p*(1-p))

## Binomial(n, p)
Use case: number of successful cell attachments out of a fixed assay count.

In [ ]:
n, p = 20, 0.35
samples_bin = rng.binomial(n, p, size=N)
xs = np.arange(0, n + 1)
pmf = binom.pmf(xs, n, p)
plot_discrete(samples_bin, xs, pmf, f'Binomial(n={n}, p={p})')
report_moments('Binomial', samples_bin, theo_mean=n*p, theo_var=n*p*(1-p))

## Categorical(π)
Use case: inflammation response class (mild / moderate / severe / critical).

In [ ]:
pi = np.array([0.15, 0.45, 0.30, 0.10])
classes = np.arange(len(pi))
samples_cat = rng.choice(classes, size=N, p=pi)

fig, ax = plt.subplots(figsize=(7, 4))
emp = np.bincount(samples_cat, minlength=len(pi)) / N
ax.bar(classes, emp, alpha=0.6, label='empirical PMF')
ax.plot(classes, pi, 'o-', color='black', label='theoretical PMF')
ax.set_title('Categorical(pi=[0.15, 0.45, 0.30, 0.10])')
ax.set_xlabel('class')
ax.set_ylabel('probability')
ax.legend()
plt.show()

values = classes
theo_mean = float(np.sum(values * pi))
theo_var = float(np.sum((values - theo_mean)**2 * pi))
report_moments('Categorical (coded as 0..K-1)', samples_cat, theo_mean, theo_var)

## Gaussian N(μ, σ²)
Use case: measurement noise on elastic modulus readings.

In [ ]:
mu, sigma = 5.0, 1.2
samples_norm = rng.normal(mu, sigma, size=N)
x_grid = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)
pdf = norm.pdf(x_grid, loc=mu, scale=sigma)
plot_continuous(samples_norm, x_grid, pdf, f'Gaussian(mu={mu}, sigma={sigma})')
report_moments('Gaussian', samples_norm, theo_mean=mu, theo_var=sigma**2)

## Multivariate Gaussian N(μ, Σ)
Use case: correlated uncertainty across two material descriptors.

In [ ]:
mu_vec = np.array([0.0, 0.0])
Sigma = np.array([[1.0, 0.7], [0.7, 1.5]])
samples_mvn = rng.multivariate_normal(mu_vec, Sigma, size=N)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(samples_mvn[:, 0], samples_mvn[:, 1], alpha=0.35, s=12)
ax.set_title('Multivariate Gaussian samples')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.axis('equal')
plt.show()

emp_mean = samples_mvn.mean(axis=0)
emp_cov = np.cov(samples_mvn.T, ddof=0)
print('Multivariate Gaussian:')
print('  empirical mean:')
print(emp_mean)
print('  theoretical mean:')
print(mu_vec)
print('  empirical covariance:')
print(emp_cov)
print('  theoretical covariance:')
print(Sigma)

## Gamma(α, β) [shape-rate]
Use case: positive, right-skewed time-to-degradation variable.

In [ ]:
alpha, beta_rate = 3.0, 1.4
scale = 1.0 / beta_rate
samples_gamma = rng.gamma(shape=alpha, scale=scale, size=N)
x_grid = np.linspace(0, np.percentile(samples_gamma, 99.5), 400)
pdf = gamma.pdf(x_grid, a=alpha, scale=scale)
plot_continuous(samples_gamma, x_grid, pdf, f'Gamma(alpha={alpha}, beta={beta_rate})')
report_moments('Gamma', samples_gamma, theo_mean=alpha/beta_rate, theo_var=alpha/(beta_rate**2))

## Beta(α, β)
Use case: uncertainty over a complication probability bounded in [0, 1].

In [ ]:
alpha, beta_param = 2.5, 5.0
samples_beta = rng.beta(alpha, beta_param, size=N)
x_grid = np.linspace(0.001, 0.999, 400)
pdf = beta.pdf(x_grid, a=alpha, b=beta_param)
plot_continuous(samples_beta, x_grid, pdf, f'Beta(alpha={alpha}, beta={beta_param})')
theo_mean = alpha / (alpha + beta_param)
theo_var = (alpha * beta_param) / (((alpha + beta_param)**2) * (alpha + beta_param + 1))
report_moments('Beta', samples_beta, theo_mean=theo_mean, theo_var=theo_var)

## Exercise reflection prompts

1. Which distributions matched theory fastest (smallest mean/variance gaps)?
2. How does increasing sample size N affect empirical-theoretical agreement?
3. For your own biomaterials dataset, which variable maps naturally to each distribution?
4. What modeling mistakes happen if support constraints are ignored (e.g., Gaussian for probabilities)?